PHẦN 1:TRẮC NGHIỆM

*CÂU 1:  Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)*

Bước 1: Tải dữ liệu lên

In [1]:
from google.colab import files
uploaded = files.upload()

Saving orders.csv to orders.csv


In [2]:
import pandas as pd

df_raw = pd.read_csv('orders.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw.copy()

In [3]:
import pandas as pd

df = pd.read_csv('orders.csv')

import pandas as pd

df = pd.read_csv('orders.csv')
print(f"Ban đầu: {len(df)} dòng")

# Bước 1: Bỏ null
df = df.dropna(subset=['order_date', 'customer_id'])
print(f"Sau bước 1 (dropna): {len(df)} dòng")

# Bước 2: Bỏ đơn cancelled và created (nếu có), returned
df = df[~df['order_status'].isin(['cancelled', 'created', 'returned'])]
print(f"Sau bước 2 (bỏ cancelled, returned, và created): {len(df)} dòng")



# Bước 3: Drop duplicate cặp (order_id, customer_id)
df = df.drop_duplicates(subset=['order_id', 'customer_id'], keep='first')
print(f"Sau bước 3 (dedup): {len(df)} dòng")

# Bước 4: Lọc customer có > 1 đơn
order_counts = df.groupby('customer_id')['order_id'].count()
valid_customers = order_counts[order_counts > 1].index
df = df[df['customer_id'].isin(valid_customers)]
print(f"Sau bước 4 (>1 đơn): {len(df)} dòng")

# Bước 5: Sort theo (customer_id, order_date)
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values(['customer_id', 'order_date']).reset_index(drop=True)

# Bước 6: Tính gap từng cặp liên tiếp trong mỗi customer
df['prev_date'] = df.groupby('customer_id')['order_date'].shift(1)
df['gap_days'] = (df['order_date'] - df['prev_date']).dt.days
gaps = df.dropna(subset=['gap_days'])['gap_days']
print(f"\nTổng số gap tính được: {len(gaps)}")

# Bước 7: Sort dãy gap tăng dần
gaps_sorted = gaps.sort_values().reset_index(drop=True)

# Bước 8: Tìm median thủ công
n = len(gaps_sorted)
if n % 2 == 1:
    median_gap = gaps_sorted[n // 2]
else:
    median_gap = (gaps_sorted[n // 2 - 1] + gaps_sorted[n // 2]) / 2

print(f"Median inter-order gap: {median_gap} ngày")

# Verify bằng pandas
print(f"Verify bằng pandas median: {gaps.median()} ngày")

Ban đầu: 646945 dòng
Sau bước 1 (dropna): 646945 dòng
Sau bước 2 (bỏ cancelled, returned, và created): 544066 dòng
Sau bước 3 (dedup): 544066 dòng
Sau bước 4 (>1 đơn): 521265 dòng

Tổng số gap tính được: 457733
Median inter-order gap: 168.0 ngày
Verify bằng pandas median: 168.0 ngày


Bài 2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price −cogs)/price?

Cách làm: Tính lợi nhuận gộp từng nhóm rồi chọn nhóm nào lợi nhuận cao nhất.

In [17]:
import pandas as pd

# Đường dẫn đến tệp đã tải lên
file_path = '/content/products.csv'

# Đọc tệp CSV vào DataFrame
df_products = pd.read_csv(file_path)

# Hiển thị 5 dòng đầu tiên của DataFrame để kiểm tra
display(df_products.head())

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [18]:
import pandas as pd

df_raw1 = pd.read_csv('products.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw1.copy()

In [19]:
print(df.shape)
print(df.dtypes)
print(df.head())

(2412, 8)
product_id        int64
product_name     object
category         object
segment          object
size             object
color            object
price           float64
cogs            float64
dtype: object
   product_id      product_name    category   segment size   color  \
0         536  SaigonFlex UC-01  Streetwear  Everyday    S   green   
1         537  SaigonFlex UC-02  Streetwear  Everyday    M  silver   
2         538  SaigonFlex UC-03  Streetwear  Everyday    L    pink   
3         539  SaigonFlex UC-04  Streetwear  Everyday   XL  yellow   
4         540  SaigonFlex UC-05  Streetwear  Everyday    S     red   

          price          cogs  
0  11059.650000   9704.842875  
1   9523.076013   5393.870254  
2  15951.633158  11371.919278  
3  15753.717299   8573.172954  
4  15766.334536  14063.570406  


In [20]:
print(df[["segment", "price", "cogs"]].isnull().sum())
print("price <= 0:", (df["price"] <= 0).sum())
print("cogs >= price:", (df["cogs"] >= df["price"]).sum())
print("cogs < 0:", (df["cogs"] < 0).sum())
print(df["segment"].value_counts())
print("Có khoảng trắng thừa:", df["segment"].str.contains(r"^\s|\s$").sum())

segment    0
price      0
cogs       0
dtype: int64
price <= 0: 0
cogs >= price: 0
cogs < 0: 0
segment
Activewear     598
Everyday       405
Performance    347
Balanced       306
Standard       262
Premium        177
All-weather    169
Trendy         148
Name: count, dtype: int64
Có khoảng trắng thừa: 0


In [16]:
df["gross_margin"] = (df["price"] - df["cogs"]) / df["price"]

result = (
    df.groupby("segment")["gross_margin"]
    .agg(
        avg_margin="mean",
        count="count",
        std_margin="std"
    )
    .sort_values("avg_margin", ascending=False)
)

print(result.round(4))
print("\n Segment cao nhất:", result.index[0])
print(f"   Avg margin: {result.iloc[0]['avg_margin']*100:.2f}%")

             avg_margin  count  std_margin
segment                                   
Standard         0.3134    262      0.1341
Premium          0.2854    177      0.1488
All-weather      0.2842    169      0.1456
Activewear       0.2656    598      0.1512
Performance      0.2636    347      0.1568
Balanced         0.2580    306      0.1495
Trendy           0.2408    148      0.1584
Everyday         0.2363    405      0.1499

 Segment cao nhất: Standard
   Avg margin: 31.34%


In [21]:
import pandas as pd

# Đường dẫn đến tệp đã tải lên
file_path = '/content/products.csv'

# Đọc tệp CSV vào DataFrame
df_products = pd.read_csv(file_path)

# Hiển thị 5 dòng đầu tiên của DataFrame để kiểm tra
display(df_products.head())

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


CÂU 3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [4]:
import pandas as pd

# Đường dẫn đến tệp đã tải lên
file_path_products = '/content/products.csv'
file_path_returns = '/content/returns.csv'

# Đọc tệp CSV vào DataFrame
df_products = pd.read_csv(file_path_products)
df_returns = pd.read_csv(file_path_returns)

print('5 dòng đầu tiên của products.csv:')
display(df_products.head())

print('\n5 dòng đầu tiên của returns.csv:')
display(df_returns.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/products.csv'

In [5]:
from google.colab import files
uploaded = files.upload()

Saving products.csv to products.csv


In [6]:
import pandas as pd

df_raw2 = pd.read_csv('products.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw2.copy()

In [7]:
from google.colab import files
uploaded = files.upload()

Saving returns.csv to returns.csv


In [8]:
import pandas as pd

df_raw3 = pd.read_csv('products.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw3.copy()

In [9]:
# Load
returns = pd.read_csv('returns.csv')
products = pd.read_csv('products.csv')
returns_raw = returns.copy()
products_raw = products.copy()

print(f"[1] returns: {len(returns)} dòng")
print(f"    products: {len(products)} dòng")
print("\nreturns null:\n", returns.isnull().sum())
print("\nproducts null:\n", products.isnull().sum())

[1] returns: 39939 dòng
    products: 2412 dòng

returns null:
 return_id          0
order_id           0
product_id         0
return_date        0
return_reason      0
return_quantity    0
refund_amount      0
dtype: int64

products null:
 product_id      0
product_name    0
category        0
segment         0
size            0
color           0
price           0
cogs            0
dtype: int64


In [10]:
# Drop thiếu dữ liệu các cột cần thiết
returns = returns.dropna(subset=['product_id', 'return_reason'])
products = products.dropna(subset=['product_id', 'category'])
print(f"\n[2] Sau dropna - returns: {len(returns)}, products: {len(products)}")


# Bước 2a: JOIN 2 bảng theo product_id
merged = returns.merge(products, on='product_id', how='inner')

# Bước 2b: Lọc category = Streetwear
streetwear = merged[merged['category'] == 'Streetwear']

# Bước 2c: Đếm từng return_reason
reason_counts = streetwear['return_reason'].value_counts()

# Bước 2d: Lấy lý do xuất hiện nhiều nhất
top_reason = reason_counts.idxmax()
top_count  = reason_counts.max()

print(reason_counts)
print(f"\nLý do nhiều nhất: '{top_reason}' với {top_count} lần")


[2] Sau dropna - returns: 39939, products: 2412
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Lý do nhiều nhất: 'wrong_size' với 7626 lần


Câu 4:  Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhấttrên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source

In [11]:
from google.colab import files
uploaded = files.upload()

Saving web_traffic.csv to web_traffic.csv


In [13]:
import pandas as pd

df_raw4 = pd.read_csv('web_traffic.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw4.copy()

In [15]:
import pandas as pd

df = pd.read_csv('web_traffic.csv')
df_raw = df.copy()
print(f"[1] Dữ liệu gốc: {len(df)} dòng")
print(df.isnull().sum())
print(f"\nCác traffic_source: {df['traffic_source'].unique()}")

# Drop thiếu dữ liệu các cột cần thiết
df = df.dropna(subset=['traffic_source', 'bounce_rate'])
print(f"\n[2] Sau dropna: {len(df)} dòng")


# Group by traffic_source, tính bounce_rate trung bình
avg_bounce = df.groupby('traffic_source')['bounce_rate'].mean() # Đã sửa từ 'traffic' thành 'df'

# Lấy nguồn có bounce_rate thấp nhất
top_source = avg_bounce.idxmin()
top_value  = avg_bounce.min()

print(avg_bounce.sort_values())
print(f"\nNguồn có bounce_rate thấp nhất: '{top_source}' = {top_value:.4f}")

[1] Dữ liệu gốc: 3652 dòng
date                        0
sessions                    0
unique_visitors             0
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64

Các traffic_source: ['organic_search' 'direct' 'referral' 'social_media' 'paid_search'
 'email_campaign']

[2] Sau dropna: 3652 dòng
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Nguồn có bounce_rate thấp nhất: 'email_campaign' = 0.0045


**Câu** 5: Tỷ ệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [16]:
from google.colab import files
uploaded = files.upload()

Saving order_items.csv to order_items.csv


In [18]:
import pandas as pd

df_raw = pd.read_csv('order_items.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw.copy()

/tmp/ipykernel_2574/2665614965.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('order_items.csv')  # bản gốc, không đụng vào


In [17]:
import pandas as pd

df = pd.read_csv('order_items.csv', low_memory=False)
total_original = len(df)
print(f"Ban đầu: {total_original} dòng")

# Bước 1: Drop duplicate theo cặp (order_id, product_id), giữ dòng đầu
df = df.drop_duplicates(subset=['order_id', 'product_id'], keep='first')
print(f"Sau bước 1 (dedup order+product): {len(df)} dòng")

# Bước 2: Bỏ dòng mà CẢ HAI promo_id VÀ promo_id_2 đều null
df = df[~(df['promo_id'].isna() & df['promo_id_2'].isna())]
print(f"Sau bước 2 (bỏ dòng không có promo nào): {len(df)} dòng")

# Tỷ lệ = số dòng còn lại / số dòng BAN ĐẦU
pct = len(df) / total_original * 100
print(f"\nTỷ lệ %: {pct:.2f}%")

Ban đầu: 714669 dòng
Sau bước 1 (dedup order+product): 714653 dòng
Sau bước 2 (bỏ dòng không có promo nào): 276309 dòng

Tỷ lệ %: 38.66%


 CÂU 6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)

In [24]:
from google.colab import files
uploaded = files.upload()

Saving customers.csv to customers (1).csv


In [25]:
import pandas as pd

df_raw = pd.read_csv('customers.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw.copy()

In [26]:
from google.colab import files

uploaded = files.upload()

Saving orders.csv to orders.csv


In [27]:
import pandas as pd

df_raw = pd.read_csv('orders.csv')
df = df_raw.copy()

# Thêm các file còn thiếu
customers = pd.read_csv('customers.csv')
orders    = pd.read_csv('orders.csv')

In [29]:
# Xem phân phối số lượng khách mỗi nhóm
print(customers_filtered['age_group'].value_counts())

# Xem signup_date trung bình theo nhóm (nhóm nào đăng ký lâu hơn)
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
print(customers.groupby('age_group')['signup_date'].min())

age_group
25-34    36342
35-44    31920
45-54    23172
18-24    17039
55+      13457
Name: count, dtype: int64
age_group
18-24   2012-02-08
25-34   2012-02-02
35-44   2012-01-17
45-54   2012-02-13
55+     2012-02-05
Name: signup_date, dtype: datetime64[ns]


In [30]:
# Đếm số order_id thực tế (không phải sum)
total_orders_real = merged.groupby('age_group')['order_id'].count()

customer_count = customers_filtered.groupby('age_group')['customer_id'].count()

summary = pd.DataFrame({
    'so_khach': customer_count,
    'tong_don': total_orders_real
}).sort_values('tong_don', ascending=False)

print(summary)

           so_khach  tong_don
age_group                    
25-34         36342    190622
35-44         31920    170368
45-54         23172    124138
18-24         17039     89057
55+           13457     72760


câu 7: Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

In [31]:
from google.colab import files

uploaded = files.upload()

Saving geography.csv to geography.csv


In [34]:
from google.colab import files

uploaded = files.upload()

Saving products.csv to products (1).csv


In [36]:
from google.colab import files

uploaded = files.upload()

Saving payments.csv to payments.csv


In [37]:
payments  = pd.read_csv('payments.csv')
orders    = pd.read_csv('orders.csv')
locations = pd.read_csv('geography.csv')

# Join payments → orders → locations
merged = payments.merge(orders[['order_id', 'order_date', 'zip']], on='order_id')
merged['order_date'] = pd.to_datetime(merged['order_date'])

# Lọc train period
train = merged[merged['order_date'] <= '2022-12-31']

# Join lấy region
train_region = train.merge(locations, on='zip', how='left')

# Tổng doanh thu theo region
result = train_region.groupby('region')['payment_value'].sum().sort_values(ascending=False)
print(result)
print(f"\nVùng doanh thu cao nhất: '{result.idxmax()}' = {result.max():,.0f}")

region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64

Vùng doanh thu cao nhất: 'East' = 7,291,150,819
